# Translate `DeepPavlov/wizard_of_wikipedia` to French and Spanish

Wizard of Wikipedia is a knowledge-grounded conversation **retrieval** dataset:
`corpus` (Wikipedia knowledge passages), `queries` (persona-grounded chit-chat about a
topic), `qrels` (relevance judgments, no text -- copied through unchanged).

**Corpus dedup.** `corpus` has `test`(19,697)/`train`(165,023)/`valid`(19,005) docs, but
`test` and `valid` doc ids are a confirmed **subset** of `train`'s (same Wikipedia
passages, reused). So the corpus is translated once (train's 165,023 docs) and reused
for all three splits -- still a big job: real Wikipedia prose, avg ~750 chars, some up
to ~136k chars.

**Query dedup.** `queries` has `train`(166,787)/`test`(8,782)/`valid`(8,806) rows, but
each conversation is stored **repeated** across ids like `query_5_0`, `query_5_1`, ...,
`query_5_8` -- verified across the full dataset that every row sharing a
`query_{conv}_*` prefix has byte-identical `persona`/`topic`/`text` (0 exceptions
found). So this notebook translates once per unique conversation (~18,430 train + 968
test + 967 valid = ~20,365 total) and replays the result across all rows in that group
-- a ~9x reduction from the raw row count.

**Register**: `corpus` is formal encyclopedic prose (no HTML/markup found); `queries`
turns are casual conversational chit-chat, `persona` a short trait sentence, `topic` a
short subject/entity name -- separate prompts are used for each.

- `gemma` — `google/gemma-4-31B-it` on `http://localhost:8088/v1`
- `qwen`  — `Qwen/Qwen3.6-27B-FP8` on `http://localhost:8000/v1`

**Setup.** This repo's `uv` environment already has `datasets`; it does not have
`openai`. Launch this notebook with the extra dependency pulled in on the fly, without
touching `pyproject.toml`:

```bash
uv run --with openai --with ipykernel jupyter lab
```

Everything is checkpointed to `translations/wow/*.jsonl`, so the notebook is safe to
interrupt and re-run — already-translated items are skipped. Given the corpus size, a
`MAX_CORPUS_DOCS` knob is provided below to test on a subset before committing to the
full 165,023 documents.


In [1]:
import json
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset
from openai import OpenAI
from tqdm.auto import tqdm


## Config

In [2]:
MODELS = {
    # "gemma": {"base_url": "http://localhost:8088/v1", "model": "google/gemma-4-31B-it"},
    "qwen": {
        "base_url": "http://localhost:8000/v1",
        "model": "Qwen/Qwen3.6-27B-FP8",
        # Qwen3 is a hybrid-thinking model: without this it emits its chain-of-thought
        # as the actual response content instead of a final answer.
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
}

LANGUAGES = {
    "fr": "French",
    "es": "Spanish",
}

QUERY_SPLITS_TO_RUN = ["valid", "test", "train"]  # smallest/fastest-feedback split first

# Corpus is translated once from "train" (test/valid doc ids are a confirmed subset).
# Set to an int (e.g. 500) to test on a sample before committing to the full 165,023 docs.
MAX_CORPUS_DOCS = None

OUT_DIR = Path("translations/wow")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 64
TEMPERATURE = 0.0


In [3]:
clients = {name: OpenAI(base_url=cfg["base_url"], api_key="EMPTY") for name, cfg in MODELS.items()}

for name, client in clients.items():
    available = [m.id for m in client.models.list().data]
    print(f"{name} ({MODELS[name]['base_url']}): serving {available}")
    assert MODELS[name]["model"] in available, (
        f"{MODELS[name]['model']} not found on {name} server; available: {available}"
    )


qwen (http://localhost:8000/v1): serving ['Qwen/Qwen3.6-27B-FP8']


## Load the dataset

In [4]:
raw_corpus = load_dataset("DeepPavlov/wizard_of_wikipedia", "corpus")
raw_queries = load_dataset("DeepPavlov/wizard_of_wikipedia", "queries")
raw_qrels = load_dataset("DeepPavlov/wizard_of_wikipedia", "qrels")

corpus_ids = {split: set(raw_corpus[split]["id"]) for split in raw_corpus}
assert corpus_ids["test"] <= corpus_ids["train"], "expected test corpus ids to be a subset of train"
assert corpus_ids["valid"] <= corpus_ids["train"], "expected valid corpus ids to be a subset of train"
print("verified: test/valid corpus doc ids are a subset of train's")

for split in raw_corpus:
    print(f"corpus/{split}: {len(raw_corpus[split])} docs")
for split in raw_queries:
    print(f"queries/{split}: {len(raw_queries[split])} rows")


verified: test/valid corpus doc ids are a subset of train's
corpus/test: 19697 docs
corpus/train: 165023 docs
corpus/valid: 19005 docs
queries/test: 8782 rows
queries/train: 166787 rows
queries/valid: 8806 rows


## Deduplicate queries by conversation

Groups rows by the `query_{conv}` id prefix and picks each group's first row as the
representative to translate -- verified against the full dataset that this is safe
(every row sharing a `query_{conv}_*` prefix has byte-identical `persona`/`topic`/
`text`, 0 exceptions). The check below re-verifies this and prints a loud warning if it
ever finds an inconsistent group (translation would then only reflect that group's
first row, not the differing ones) -- it does not silently proceed past a mismatch
without telling you.


In [5]:
def conv_id_of(row_id):
    return row_id.rsplit("_", 1)[0]


def group_queries(dataset):
    groups = {}
    for row in dataset:
        groups.setdefault(conv_id_of(row["id"]), []).append(row)
    return groups


query_groups = {split: group_queries(raw_queries[split]) for split in raw_queries}

inconsistent = []
for split, groups in query_groups.items():
    for conv_id, rows in groups.items():
        sigs = {(r["persona"], r["topic"], tuple(t["content"] for t in r["text"])) for r in rows}
        if len(sigs) > 1:
            inconsistent.append((split, conv_id))
    print(f"queries/{split}: {len(raw_queries[split])} rows -> {len(groups)} unique conversations")

if inconsistent:
    print(f"WARNING: {len(inconsistent)} conversation groups are NOT identical across their rows "
          f"(e.g. {inconsistent[:5]}) -- those groups' translations will only reflect their first "
          f"row. Re-run this cell's check after investigating if this count is unexpectedly high.")
else:
    print("verified: every conversation group is identical across its rows -- safe to dedupe")


queries/test: 8782 rows -> 968 unique conversations
queries/train: 166787 rows -> 18430 unique conversations
queries/valid: 8806 rows -> 967 unique conversations
verified: every conversation group is identical across its rows -- safe to dedupe


## Translation prompts

Separate system prompts for the encyclopedic `corpus` register vs. the casual
conversational `queries` register (turns/persona/topic).


In [6]:
CORPUS_EXAMPLES = {
    "fr": [
        (
            "A pharmacy technician is a health care provider who performs pharmacy-related functions.",
            "Un technicien en pharmacie est un professionnel de sante qui exerce des fonctions liees a la pharmacie.",
        ),
    ],
    "es": [
        (
            "A pharmacy technician is a health care provider who performs pharmacy-related functions.",
            "Un tecnico de farmacia es un profesional de la salud que realiza funciones relacionadas con la farmacia.",
        ),
    ],
}

CORPUS_SYSTEM_PROMPT = (
    "You are a professional translator localizing Wikipedia knowledge-base passages for "
    "a conversational search system. Translate the text from English into {lang_name}, "
    "producing formal, accurate, encyclopedic {lang_name} prose. Keep proper nouns "
    "(people, places, works, organizations) as their correct {lang_name} form where a "
    "standard one exists, otherwise leave them as written. Preserve facts, numbers, "
    "and dates exactly. Do not add, remove, summarize, or explain anything -- translate "
    "the full passage. Reply with ONLY the translation: no quotes, no notes.\n\n"
    "Examples:\n{examples_block}"
)


def build_corpus_examples_block(lang_code):
    lines = [f"EN: {en}\n{lang_code.upper()}: {es}" for en, es in CORPUS_EXAMPLES.get(lang_code, [])]
    return "\n\n".join(lines)


QUERY_EXAMPLES = {
    "fr": [
        (
            "I think science fiction is an amazing genre for anything.",
            "Je trouve que la science-fiction est un genre formidable pour a peu pres tout.",
        ),
        ("Science fiction", "Science-fiction"),
        ("my mother met elvis.", "ma mere a rencontre elvis."),
    ],
    "es": [
        (
            "I think science fiction is an amazing genre for anything.",
            "Creo que la ciencia ficcion es un genero increible para casi cualquier cosa.",
        ),
        ("Science fiction", "Ciencia ficcion"),
        ("my mother met elvis.", "mi madre conocio a elvis."),
    ],
}

QUERY_SYSTEM_PROMPT = (
    "You are a professional translator localizing a casual chit-chat conversation "
    "between two people discussing a topic (persona-grounded dialogue). Translate the "
    "given text from English into {lang_name}. It may be one line of dialogue, a short "
    "persona trait sentence, or a short topic/subject name -- in every case, keep the "
    "same meaning, tone, and casual register, using natural {lang_name}. Keep proper "
    "nouns (people, places, titles of works) as their correct {lang_name} form where one "
    "exists. Do not add, remove, or explain anything. "
    "Reply with ONLY the translation: no quotes, no notes, no alternatives.\n\n"
    "Examples:\n{examples_block}"
)


def build_query_examples_block(lang_code):
    lines = [f"EN: {en}\n{lang_code.upper()}: {es}" for en, es in QUERY_EXAMPLES.get(lang_code, [])]
    return "\n\n".join(lines)


THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
REASONING_MARKERS = re.compile(
    r"^\s*(here'?s a thinking process|let'?s (think|analyze)|step \d|\d+\.\s+\*\*)",
    re.IGNORECASE,
)


def clean_translation(raw_out):
    out = THINK_RE.sub("", raw_out).strip()
    return out.strip('"').strip("'").strip()


def translate_with_prompt(client, model, text, lang_code, system_prompt_template, examples_block, extra_body=None, temperature=TEMPERATURE, max_retries=5):
    lang_name = LANGUAGES[lang_code]
    system = system_prompt_template.format(lang_name=lang_name, examples_block=examples_block)
    messages = [{"role": "system", "content": system}, {"role": "user", "content": text}]
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=temperature, extra_body=extra_body or {},
            )
            out = clean_translation(resp.choices[0].message.content)
            if out and not REASONING_MARKERS.search(out):
                return out
            last_err = RuntimeError(f"looks like leaked reasoning: {out[:120]!r}")
        except Exception as e:  # noqa: BLE001
            last_err = e
        time.sleep(min(2 ** attempt, 20))
    raise RuntimeError(f"Translation failed for {text!r}: {last_err}")


def translate_corpus_item(client, model, text, lang_code, extra_body=None):
    return translate_with_prompt(
        client, model, text, lang_code, CORPUS_SYSTEM_PROMPT, build_corpus_examples_block(lang_code), extra_body,
    )


def translate_query_item(client, model, text, lang_code, extra_body=None):
    return translate_with_prompt(
        client, model, text, lang_code, QUERY_SYSTEM_PROMPT, build_query_examples_block(lang_code), extra_body,
    )


## Checkpointed, concurrent translation of the corpus (train docs only)

Each doc contributes 2 units: `title` and `text`.


In [7]:
def translate_corpus(lang_code, model_key, position=None):
    client = clients[model_key]
    model = MODELS[model_key]["model"]
    extra_body = MODELS[model_key].get("extra_body", {})

    docs = raw_corpus["train"]
    if MAX_CORPUS_DOCS is not None:
        docs = docs.select(range(min(MAX_CORPUS_DOCS, len(docs))))

    units = []
    for row in docs:
        units.append((row["id"], "title", row["title"]))
        units.append((row["id"], "text", row["text"]))

    out_path = OUT_DIR / f"corpus_{lang_code}_{model_key}.jsonl"
    done = {}
    if out_path.exists():
        with out_path.open() as f:
            for line in f:
                row = json.loads(line)
                done[(row["doc_id"], row["kind"])] = row["translated"]

    todo = [u for u in units if (u[0], u[1]) not in done]
    print(f"[corpus/{lang_code}/{model_key}] {len(done)} cached, {len(todo)} to translate via {model}")

    if todo:
        with out_path.open("a") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(translate_corpus_item, client, model, text, lang_code, extra_body): (doc_id, kind)
                for doc_id, kind, text in todo
            }
            bar = tqdm(as_completed(futures), total=len(futures), desc=f"{model_key}: corpus/{lang_code}", position=position, leave=True)
            for fut in bar:
                key = futures[fut]
                translated = fut.result()
                row = {"doc_id": key[0], "kind": key[1], "translated": translated}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
                done[key] = translated

    return done


## Checkpointed, concurrent translation of queries (unique conversations only)

Translates `persona`, `topic`, and every turn once per unique conversation. Units are
keyed by `(conv_id, kind, turn_idx)`.


In [8]:
def translate_queries_conversations(split_name, lang_code, model_key, position=None):
    client = clients[model_key]
    model = MODELS[model_key]["model"]
    extra_body = MODELS[model_key].get("extra_body", {})

    groups = query_groups[split_name]
    units = []
    for conv_id, rows in groups.items():
        rep = rows[0]  # verified identical across the group (or handled below if not)
        units.append((conv_id, "persona", 0, rep["persona"]))
        units.append((conv_id, "topic", 0, rep["topic"]))
        for turn_idx, turn in enumerate(rep["text"]):
            units.append((conv_id, "turn", turn_idx, turn["content"]))

    out_path = OUT_DIR / f"queries_{split_name}_{lang_code}_{model_key}.jsonl"
    done = {}
    if out_path.exists():
        with out_path.open() as f:
            for line in f:
                row = json.loads(line)
                done[(row["conv_id"], row["kind"], row["turn_idx"])] = row["translated"]

    todo = [u for u in units if (u[0], u[1], u[2]) not in done]
    print(f"[queries/{split_name}/{lang_code}/{model_key}] {len(done)} cached, {len(todo)} to translate via {model}")

    if todo:
        with out_path.open("a") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(translate_query_item, client, model, text, lang_code, extra_body): (conv_id, kind, turn_idx)
                for conv_id, kind, turn_idx, text in todo
            }
            bar = tqdm(as_completed(futures), total=len(futures), desc=f"{model_key}: queries/{split_name}/{lang_code}", position=position, leave=True)
            for fut in bar:
                key = futures[fut]
                translated = fut.result()
                row = {"conv_id": key[0], "kind": key[1], "turn_idx": key[2], "translated": translated}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
                done[key] = translated

    return done


## Smoke test

Translate a small corpus sample and a few conversations with both models before committing to the full run.

In [9]:
smoke_corpus_docs = MAX_CORPUS_DOCS
MAX_CORPUS_DOCS = 5
smoke_conv_ids = list(query_groups["valid"])[:3]
smoke_query_groups_backup = query_groups["valid"]
query_groups["valid"] = {cid: smoke_query_groups_backup[cid] for cid in smoke_conv_ids}

for lang_code in LANGUAGES:
    for model_key in MODELS:
        corpus_result = translate_corpus(lang_code, model_key)
        for row in raw_corpus["train"].select(range(2)):
            print(f"[corpus/{lang_code}/{model_key}] {row['title']!r} -> {corpus_result[(row['id'], 'title')]!r}")
        query_result = translate_queries_conversations("valid", lang_code, model_key)
        for conv_id in smoke_conv_ids[:1]:
            rep = smoke_query_groups_backup[conv_id][0]
            print(f"[queries/{lang_code}/{model_key}] TOPIC: {rep['topic']!r} -> {query_result[(conv_id, 'topic', 0)]!r}")
            print(f"[queries/{lang_code}/{model_key}] PERSONA: {rep['persona']!r} -> {query_result[(conv_id, 'persona', 0)]!r}")
        print()

query_groups["valid"] = smoke_query_groups_backup
MAX_CORPUS_DOCS = smoke_corpus_docs


[corpus/fr/qwen] 0 cached, 10 to translate via Qwen/Qwen3.6-27B-FP8


qwen: corpus/fr:   0%|          | 0/10 [00:00<?, ?it/s]

[corpus/fr/qwen] 'Hyperspace (science fiction)' -> 'Hyperspace (science-fiction)'
[corpus/fr/qwen] 'Science fiction' -> 'La science-fiction'
[queries/valid/fr/qwen] 0 cached, 34 to translate via Qwen/Qwen3.6-27B-FP8


qwen: queries/valid/fr:   0%|          | 0/34 [00:00<?, ?it/s]

[queries/fr/qwen] TOPIC: 'Red' -> 'Rouge'
[queries/fr/qwen] PERSONA: 'my favorite color is red.' -> 'ma couleur préférée est le rouge.'

[corpus/es/qwen] 0 cached, 10 to translate via Qwen/Qwen3.6-27B-FP8


qwen: corpus/es:   0%|          | 0/10 [00:00<?, ?it/s]

[corpus/es/qwen] 'Hyperspace (science fiction)' -> 'Hiperspacio (ciencia ficción)'
[corpus/es/qwen] 'Science fiction' -> 'Ciencia ficción'
[queries/valid/es/qwen] 0 cached, 34 to translate via Qwen/Qwen3.6-27B-FP8


qwen: queries/valid/es:   0%|          | 0/34 [00:00<?, ?it/s]

[queries/es/qwen] TOPIC: 'Red' -> 'Rojo'
[queries/es/qwen] PERSONA: 'my favorite color is red.' -> 'mi color favorito es el rojo.'



## Full run

Same gemma/qwen-in-parallel pattern as the earlier notebooks: each model translates the
corpus, then works through `QUERY_SPLITS_TO_RUN` for queries, on its own thread with its
own progress-bar row.


In [ ]:
def run_model_jobs(model_key, position):
    results = {"corpus": {}, "queries": {}}
    for lang_code in LANGUAGES:
        results["corpus"][lang_code] = translate_corpus(lang_code, model_key, position=position)
    for split_name in QUERY_SPLITS_TO_RUN:
        for lang_code in LANGUAGES:
            results["queries"][(split_name, lang_code)] = translate_queries_conversations(
                split_name, lang_code, model_key, position=position
            )
    return model_key, results


translated_corpus = {}
translated_queries = {}
with ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    futures = {ex.submit(run_model_jobs, model_key, position): model_key for position, model_key in enumerate(MODELS)}
    for fut in as_completed(futures):
        model_key, results = fut.result()
        for lang_code, lookup in results["corpus"].items():
            translated_corpus[(lang_code, model_key)] = lookup
        for (split_name, lang_code), lookup in results["queries"].items():
            translated_queries[(split_name, lang_code, model_key)] = lookup


[corpus/fr/qwen] 10 cached, 330036 to translate via Qwen/Qwen3.6-27B-FP8


qwen: corpus/fr:   0%|          | 0/330036 [00:00<?, ?it/s]

## Assemble translated datasets

`corpus` docs are reconstructed once per (lang, model) and reused for `train`/`test`/
`valid` (subset relationship). `queries` rows are reconstructed by replaying each
conversation's shared translation across every row in its group. `qrels` is copied
through unchanged.


In [ ]:
final_corpus = {}
for lang_code in LANGUAGES:
    for model_key in MODELS:
        lookup = translated_corpus[(lang_code, model_key)]
        by_id = {
            row["id"]: {"id": row["id"], "title": lookup[(row["id"], "title")], "text": lookup[(row["id"], "text")]}
            for row in raw_corpus["train"].select(range(min(MAX_CORPUS_DOCS, len(raw_corpus["train"])) if MAX_CORPUS_DOCS else len(raw_corpus["train"])))
        }
        dd = DatasetDict()
        for split_name in raw_corpus:
            rows = [by_id[doc_id] for doc_id in raw_corpus[split_name]["id"] if doc_id in by_id]
            dd[split_name] = Dataset.from_list(rows)
        final_corpus[(lang_code, model_key)] = dd
        print("corpus", lang_code, model_key, {s: len(dd[s]) for s in dd})


In [ ]:
final_queries = {}
for lang_code in LANGUAGES:
    for model_key in MODELS:
        dd = DatasetDict()
        for split_name in QUERY_SPLITS_TO_RUN:
            lookup = translated_queries[(split_name, lang_code, model_key)]
            rows = []
            for row in raw_queries[split_name]:
                conv_id = conv_id_of(row["id"])
                text = [
                    {"content": lookup[(conv_id, "turn", i)], "role": turn["role"]}
                    for i, turn in enumerate(row["text"])
                ]
                rows.append({
                    "id": row["id"],
                    "text": text,
                    "persona": lookup[(conv_id, "persona", 0)],
                    "topic": lookup[(conv_id, "topic", 0)],
                })
            dd[split_name] = Dataset.from_list(rows)
        final_queries[(lang_code, model_key)] = dd
        print("queries", lang_code, model_key, {s: len(dd[s]) for s in dd})


## Spot-check quality

In [ ]:
lang_code = "es"
model_key = "gemma"
row = final_corpus[(lang_code, model_key)]["test"][0]
print("CORPUS TITLE:", row["title"])
print("CORPUS TEXT:", row["text"][:300])
print()
qrow = final_queries[(lang_code, model_key)]["valid"][0]
print("TOPIC:", qrow["topic"])
print("PERSONA:", qrow["persona"])
for turn in qrow["text"][:3]:
    print(f"  [{turn['role']}]", turn["content"][:150])


## Save

Saves corpus/queries per (language, model); `qrels` is copied through as-is (no text,
language-independent). Pushing to the Hub is left commented out.


In [ ]:
SAVE_DIR = Path("translations/wow_final")
for (lang_code, model_key), dd in final_corpus.items():
    dd.save_to_disk(str(SAVE_DIR / f"{lang_code}-{model_key}-corpus"))
for (lang_code, model_key), dd in final_queries.items():
    dd.save_to_disk(str(SAVE_DIR / f"{lang_code}-{model_key}-queries"))
raw_qrels.save_to_disk(str(SAVE_DIR / "qrels"))

# repo_id = "<your-username>/wizard-of-wikipedia-mt"
# for (lang_code, model_key), dd in {**final_corpus}.items():
#     dd.push_to_hub(repo_id, config_name=f"{lang_code}-{model_key}-corpus")
# for (lang_code, model_key), dd in {**final_queries}.items():
#     dd.push_to_hub(repo_id, config_name=f"{lang_code}-{model_key}-queries")
# raw_qrels.push_to_hub(repo_id, config_name="qrels")
